In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from icdn import PanelSchema
from icdn.data.splits import TemporalSplitter

ROOT = Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.benchmarks.features import ICDNFeaturePipeline
from src.benchmarks.pairs import PairDatasetBuilder
from src.benchmarks.demand_mlp import DemandMLPPipeline

DATASETS = {
    "dunnhumby": {
        "path": ROOT / "data" / "dunnhumby" / "panel" / "dunnhumby_icdn_panel.parquet",
        "schema": PanelSchema(category="category", brand="brand", style="style"),
    },
    "walmart": {
        "path": ROOT / "data" / "M5-walmart" / "panel" / "m5_icdn_panel.parquet",
        "schema": PanelSchema(category="category"),
    },
    "one_c": {
        "path": ROOT / "data" / "predict-future-sales-1c" / "panel" / "1c_icdn_panel.parquet",
        "schema": PanelSchema(category="category"),
    },
}

def run_one(name, spec):
    panel = pd.read_parquet(spec["path"])
    panel = panel[(panel["price"] > 0) & (panel["units"] > 0)].copy()
    print(f"\n=== {name} === {panel.shape}  "
          f"products={panel['product_code'].nunique()}  "
          f"stores={panel['store_code'].nunique()}")

    splitter = TemporalSplitter(period_col="week_id")
    train_raw, val_raw = splitter.single_split(panel, train_frac=0.8)

    feats = ICDNFeaturePipeline(schema=spec["schema"])
    train = feats.fit(train_raw).transform(train_raw)
    val = feats.transform_val(val_raw)
    print("controls:", len(feats.control_cols), feats.control_cols)

    builder = PairDatasetBuilder(feats.control_cols)
    train_pairs = builder.build(train)
    val_pairs = builder.build(val)
    print("pairs train/val:", train_pairs.shape, val_pairs.shape)

    mlp = DemandMLPPipeline(feats.control_cols)  # defaults = MLPConfig original

    metrics, elast = mlp.run(train_pairs, val_pairs)
    g = elast.assign(abs_err=lambda d: (d["y_true_i"] - d["y_hat_i"]).abs())
    summary = g.groupby(["store_code", "pair_id", "product_i", "product_j"], as_index=False).agg(
        own_elasticity=("own_elasticity", "mean"),
        cross_elasticity=("cross_elasticity", "mean"),
        mae_val=("abs_err", "mean"),
        n_val=("week_id", "size"),
    )
    print("own  mean/min/max", summary["own_elasticity"].mean(), summary["own_elasticity"].min(), summary["own_elasticity"].max())
    print("cross mean/min/max", summary["cross_elasticity"].mean(), summary["cross_elasticity"].min(), summary["cross_elasticity"].max())
    return summary

for name, spec in DATASETS.items():
    run_one(name, spec)


=== dunnhumby === (3216, 8)  products=10  stores=20
controls: 32 ['period_rank', 'sin_52', 'cos_52', 'sin_26', 'cos_26', 'sin_13', 'cos_13', 'promo_intensity', 'periods_seen_product', 'periods_seen_store_product', 'lag_1', 'miss_lag_1', 'lag_2', 'miss_lag_2', 'lag_4', 'miss_lag_4', 'roll_4', 'miss_roll_4', 'roll_13', 'miss_roll_13', 'promo', 'n_neighbors', 'neighbor_promo_share', 'n_same_brand_neighbors', 'same_brand_promo_share', 'neighbor_lag_1', 'miss_neighbor_lag_1', 'neighbor_roll_4', 'miss_neighbor_roll_4', 'n_new_neighbors', 'share_new_neighbors', 'assortment_size']
pairs train/val: (5104, 40) (1460, 40)
groups ok/error/skipped: 28 0 564
own  mean/min/max -0.17805970224676784 -4.7965318626073135 0.7901078348698994
cross mean/min/max -0.2739987802267601 -5.2339852908708 0.9630569040883621
val MAE/RMSE 1.8412174733579152 2.114219835719406

=== walmart === (47904, 7)  products=20  stores=10
controls: 30 ['period_rank', 'sin_52', 'cos_52', 'sin_26', 'cos_26', 'sin_13', 'cos_13', 'p

KeyboardInterrupt: 